# Multilingual Phishing Detection

English and Hindi email-classification experiments using engineered features, XGBoost, SMOTE, and SHAP.

See the repository README for dependencies, required data, and current limitations.


In [ ]:
#main code########

#To calculate the feature weightage for each row and categorize it as "less suspicious," "more suspicious," or "non-suspicious,"
#you can use the trained model to compute feature contributions (SHAP values)
#or use the feature importance to create a weighted scoring system for each row. Here’s the modified code to accomplish this:
#Calculate Feature Contributions: Use SHAP (SHapley Additive exPlanations) to explain the model's predictions for each row.
#Categorize Based on Contributions: Sum the contributions for each feature in each row and assign categories based on a predefined threshold.

#threshold found by manual method by understanding the suspicion score.
import pandas as pd
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
from dateutil import parser
import re
import whois
import datetime
from datetime import datetime as dt

# Define phishing keywords and trusted domains
PHISHING_KEYWORDS = ["password", "urgent", "click here", "verify", "login", "account", "bank", "free", "offer", "win"]
trusted_domains = ['gmail.com', 'yahoo.com', 'hotmail.com']

# Function to calculate feature contributions using SHAP
def calculate_shap_values(model, X):
    explainer = shap.Explainer(model)
    shap_values = explainer(X)
    shap_df = pd.DataFrame(shap_values.values, columns=X.columns)

    # Calculate a score for each row based on SHAP values
    shap_df['suspicion_score'] = shap_df.sum(axis=1)

    # Categorize each row based on the calculated score
    # shap_df['suspicion_level'] = shap_df['suspicion_score'].apply(categorize_suspicion)

    return shap_df

# Preprocessing
def preprocess_data(df):
    df['sender'] = df['sender'].fillna('unknown').astype(str)
    df['subject'] = df['subject'].fillna('')
    df['body'] = df['body'].fillna('')
    df['urls'] = df['urls'].fillna('').astype(str)

    df['domain'] = df['sender'].apply(extract_domain)
    df['domain_suspicion'] = df['domain'].apply(lambda x: classify_domain(x, trusted_domains))
    df['time_suspicion'] = df['date'].apply(classify_time)
    df['keyword_suspicion_subject'] = df['subject'].apply(has_phishing_keywords)
    df['keyword_suspicion_body'] = df['body'].apply(has_phishing_keywords)
    df['url_suspicion'] = df['urls'].apply(lambda x: 1 if x else 0)

    df['keyword_suspicion'] = df[['keyword_suspicion_subject', 'keyword_suspicion_body']].max(axis=1)

    # Calculate domain age and categorize it
    df['domain_age'] = df['domain'].apply(get_domain_age)
    df['domain_age_suspicion'] = df['domain_age'].apply(categorize_domain_age)

    df['subject_length'] = df['subject'].apply(email_length)
    df['body_length'] = df['body'].apply(email_length)
    df['subject_hyperlink_count'] = df['subject'].apply(count_hyperlinks)
    df['body_hyperlink_count'] = df['body'].apply(count_hyperlinks) #idhar
    df['subject_sentiment'] = df['subject'].apply(analyze_sentiment)
    df['body_sentiment'] = df['body'].apply(analyze_sentiment)
    df['sender_reputation'] = df['sender'].apply(sender_reputation)

    return df

# Function to extract domain from email
def extract_domain(email):
    try:
        if isinstance(email, str):
            return email.split('@')[-1]
        else:
            return 'unknown'
    except Exception as e:
        print(f"Error extracting domain from {email}: {e}")
        return 'unknown'

# Function to classify the domain as suspicious or not
def classify_domain(domain, trusted_domains):
    try:
        if domain in trusted_domains:
            return 'less suspicious'
        elif any(trusted in domain for trusted in trusted_domains):
            return 'medium suspicious'
        else:
            return 'highly suspicious'
    except Exception as e:
        print(f"Error classifying domain {domain}: {e}")
        return 'unknown'

# Function to classify time of email
def classify_time(date_string):
    try:
        if pd.isna(date_string):
            return 0
        email_time = parser.parse(date_string)
        if email_time.hour >= 0 and email_time.hour <= 5:
            return 1
        elif email_time.hour >= 22 or email_time.hour <= 7:  # Late night/early morning emails
            return 0.5
        else:
            return 0
    except Exception as e:
        print(f"Error parsing time from {date_string}: {e}")
        return 0

# Function to check for phishing keywords
def has_phishing_keywords(text):
    try:
        if isinstance(text, str):
            for keyword in PHISHING_KEYWORDS:
                if keyword.lower() in text.lower() :
                    return 1  # 1 if phishing keyword present
        return 0  # 0 if not present
    except Exception as e:
        print(f"Error checking phishing keywords in {text}: {e}")
        return 0  # Assume no phishing keyword found on error

# Function to check if the email has a URL
def contains_url(url_flag):
    return url_flag

# Function to calculate email length
def email_length(text):
    try:
      if isinstance(text, str):
          return len(text)
      else:
          return 0
    except Exception as e:
        print(f"Error calculating email length: {e}")
        return 0

# Function for word tokenization
def tokenize_text(text):
    try:
        text = clean_text(text)
        tokens = word_tokenize(text)
        return tokens
    except Exception as e:
        print(f"Error tokenizing text {text}: {e}")
        return []

# Function to count hyperlinks
def count_hyperlinks(text):
    try:
        return len(re.findall(r'http[s]?://', text))
    except Exception as e:
        # print(f"Error counting hyperlinks in {text}: {e}")
        return 0

# Function for sentiment analysis
def analyze_sentiment(text):
    try:
        analysis = TextBlob(text)
        return analysis.sentiment.polarity  # Returns a value between -1 (negative) and 1 (positive)
    except:
        return 0

# Function to determine sender reputation based on domains
def sender_reputation(sender_email):
    common_domains = ['gmail.com', 'yahoo.com', 'hotmail.com']
    suspicious_domains = ['.ru', '.cn', '.xyz', '.top', '.icu', '.info', '.tk', '.ml', '.ga', '.cf']

    if any(domain in sender_email for domain in common_domains):
        return 0  # Less suspicious
    elif any(domain in sender_email for domain in suspicious_domains):
        return 1  # Highly suspicious
    else:
        return 0.5  # Neutral

# Function to check for phishing-related keywords
def keyword_suspicion(text):
    phishing_keywords = ['urgent', 'free', 'winner', 'congratulations', 'claim', 'prize', 'click', 'password', 'login', 'verify']
    return any(keyword in text.lower() for keyword in phishing_keywords)

# Function to get domain age in months
def get_domain_age(domain):
    """
    Function to calculate the domain age in months.
    Returns None if the domain age cannot be determined.
    """
    try:
        domain_info = whois.whois(domain)
        creation_date = domain_info.creation_date

        if isinstance(creation_date, list):  # Sometimes it's returned as a list
            creation_date = creation_date[0]

        if creation_date is None:
            return None

        # Calculate domain age in months
        domain_age = (dt.now() - creation_date).days / 30
        return domain_age
    except Exception as e:
        print(f"Error fetching data for domain {domain}: {e}")
        return None

# Function to categorize domain age
def categorize_domain_age(domain_age):
    """
    Classify domain age:
    - None or less than 6 months: Highly suspicious (2)
    - 6 months or more: Less suspicious (0)
    """
    if domain_age is None or domain_age < 6:
        return 2  # Highly suspicious
    return 0  # Less suspicious

# Wrap the entire pipeline in a try-except block
try:
    # Load the dataset
    df = pd.read_excel('/content/Cleaned_English_Dataset.xlsx')

    # Preprocess the data (same as before)
    df = preprocess_data(df)
    df.fillna(0, inplace=True)  # Replace NaN with 0

    # Prepare the dataset for modeling
    X = df[['domain_age_suspicion', 'time_suspicion', 'keyword_suspicion',
             'subject_length', 'body_length', 'subject_hyperlink_count',
             'body_hyperlink_count', 'subject_sentiment', 'body_sentiment']]
    y = df['label']

    # Address class imbalance using SMOTE
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)

    # Split the resampled data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.3, random_state=42, stratify=y_res)

    # Train an XGBoost classifier
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]))
    model.fit(X_train, y_train)

    # Calculate feature contributions and categorize suspicion levels
    shap_df = calculate_shap_values(model, X)

    # Combine the SHAP values with the original data for better analysis
    result_df = pd.concat([df, shap_df[['suspicion_level']]], axis=1)

    # Display the first few rows of the result
    print(result_df.head())

except Exception as e:
    print(f"An error occurred during processing: {e}")


In [ ]:
new_data = pd.DataFrame([{
    'sender': 'example@freemoney.win',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 03:00:00',
    'subject': 'Claim your FREE reward now!',
    'body': 'Click here to verify your account and win big prizes!',
    'urls': 'http://freemoney.win/claim'
}])

nd = preprocess_data(new_data)
nd.fillna(0, inplace=True)

nd_X = nd[['domain_age_suspicion', 'time_suspicion', 'keyword_suspicion',
             'subject_length', 'body_length', 'subject_hyperlink_count',
             'body_hyperlink_count', 'subject_sentiment', 'body_sentiment']]
prediction = model.predict(nd_X)[0]
probability = model.predict_proba(nd_X)[0][1]  # Probability of being suspicious
print(prediction)
print(probability)

# Calculate feature contributions and categorize suspicion levels
shap_df = calculate_shap_values(model, nd_X)

# Combine the SHAP values with the original data for better analysis
nd = pd.concat([nd, shap_df[['suspicion_level']]], axis=1)
print(nd)

In [ ]:
# NOT REQUIRED
# Multilingual Phishing Detection Pipeline (Hindi + English)*******
# =========================

import pandas as pd
import numpy as np
import re
from collections import Counter
from textblob import TextBlob
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import shap

# -----------------------------
# 1️⃣ Load Datasets
# -----------------------------
df_hi = pd.read_csv("/content/hindi_emails_merged.csv")
df_en = pd.read_excel("/content/Cleaned_English_Dataset.xlsx")

# -----------------------------
# 2️⃣ Shared Helper Functions
# -----------------------------
trusted_domains = ['gmail.com','yahoo.com','hotmail.com']

def extract_domain(email):
    try:
        return email.split('@')[-1] if isinstance(email,str) else 'unknown'
    except:
        return 'unknown'

def classify_domain(domain):
    if domain in trusted_domains: return 0
    suspicious_tlds = ['.ru','.cn','.xyz','.top','.icu','.info','.tk','.ml','.ga','.cf']
    if any(tld in domain for tld in suspicious_tlds): return 2
    return 1

def classify_time(date_string):
    try:
        hour = pd.to_datetime(date_string).hour
        return 1 if 0<=hour<=5 or hour>=22 else 0
    except: return 0

def email_length(text): return len(text) if isinstance(text,str) else 0
def count_hyperlinks(text): return len(re.findall(r'http[s]?://', str(text))) if isinstance(text,str) else 0
def analyze_sentiment(text):
    try: return TextBlob(str(text)).sentiment.polarity
    except: return 0

def sender_reputation(sender_email):
    if any(domain in sender_email for domain in trusted_domains): return 0
    suspicious_domains = ['.ru','.cn','.xyz','.top','.icu','.info','.tk','.ml','.ga','.cf']
    if any(domain in sender_email for domain in suspicious_domains): return 1
    return 0.5

def has_keywords(text, keyword_list):
    try:
        text = str(text).lower()
        for word in keyword_list:
            if word.lower() in text: return 1
        return 0
    except: return 0

# -----------------------------
# 3️⃣ Extract Dynamic Phishing Keywords
# -----------------------------
def extract_keywords(df, top_n=60):
    if 'label' in df.columns:
        phishing_texts = df[df['label']==1]['subject'].astype(str).tolist() + \
                         df[df['label']==1]['body'].astype(str).tolist()
    else:
        phishing_texts = df['subject'].astype(str).tolist() + df['body'].astype(str).tolist()
    words = []
    for text in phishing_texts: words.extend(re.findall(r'\w+', text))
    return [w for w,_ in Counter(words).most_common(top_n)]

PHISHING_KEYWORDS_HINDI = list(set(extract_keywords(df_hi,60) + [
    "पासवर्ड","तुरंत","यहाँ क्लिक करें","सत्यापित करें","लॉगिन","खाता",
    "बैंक","फ्री","ऑफ़र","जीतें","इनाम","अपडेट","भुगतान","रिचार्ज","सुरक्षा","संदेहास्पद","डाउनलोड"
]))
PHISHING_KEYWORDS_EN = list(set(extract_keywords(df_en,60) + [
    "lottery","prize","win","bonus","free","offer","password","verify",
    "account","click","login","update","cash","otp","urgent"
]))

# -----------------------------
# 4️⃣ Preprocessing Function
# -----------------------------
def preprocess_data(df, language='hi'):
    df = df.copy()
    df['sender'] = df['sender'].fillna('unknown').astype(str)
    df['subject'] = df['subject'].fillna('').astype(str)
    df['body'] = df['body'].fillna('').astype(str)
    if 'urls' not in df.columns:
        df['urls'] = ''
    else:
        df['urls'] = df['urls'].fillna('').astype(str)

    df['domain'] = df['sender'].apply(extract_domain)
    df['domain_suspicion'] = df['domain'].apply(classify_domain)
    df['time_suspicion'] = df['date'].apply(classify_time)

    df['keyword_suspicion_subject'] = df['subject'].apply(
        lambda x: has_keywords(x, PHISHING_KEYWORDS_HINDI if language=='hi' else PHISHING_KEYWORDS_EN)
    )
    df['keyword_suspicion_body'] = df['body'].apply(
        lambda x: has_keywords(x, PHISHING_KEYWORDS_HINDI if language=='hi' else PHISHING_KEYWORDS_EN)
    )
    df['keyword_suspicion'] = df[['keyword_suspicion_subject','keyword_suspicion_body']].max(axis=1)

    df['url_suspicion'] = df['urls'].apply(lambda x: 1 if x else 0)
    df['subject_length'] = df['subject'].apply(email_length)
    df['body_length'] = df['body'].apply(email_length)
    df['subject_hyperlink_count'] = df['subject'].apply(count_hyperlinks)
    df['body_hyperlink_count'] = df['body'].apply(count_hyperlinks)
    df['subject_sentiment'] = df['subject'].apply(analyze_sentiment)
    df['body_sentiment'] = df['body'].apply(analyze_sentiment)
    df['sender_reputation'] = df['sender'].apply(sender_reputation)

    if 'label' in df.columns:
        if df['label'].dtype==object:
            df['label'] = df['label'].map({'ham':0,'spam':1})

    return df

# -----------------------------
# 5️⃣ Train Model Function
# -----------------------------
def train_model(df, language='hi'):
    df = preprocess_data(df, language)
    df.fillna(0, inplace=True)

    features = ['domain_suspicion','time_suspicion','keyword_suspicion',
                'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
                'subject_sentiment','body_sentiment','sender_reputation']
    X = df[features]
    y = df['label']

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)

    scale_pos_weight = y_res.value_counts().iloc[0] / y_res.value_counts().iloc[1]

    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                          scale_pos_weight=scale_pos_weight)
    model.fit(X_res, y_res)
    return model, X

# -----------------------------
# 6️⃣ SHAP + Suspicion Table
# -----------------------------
def build_final_output(df, model, language='hi'):
    df_proc = preprocess_data(df, language)
    features = ['domain_suspicion','time_suspicion','keyword_suspicion',
                'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
                'subject_sentiment','body_sentiment','sender_reputation']
    X = df_proc[features]

    explainer = shap.Explainer(model)
    shap_values = explainer(X)
    shap_df = pd.DataFrame(shap_values.values, columns=X.columns)
    shap_df['suspicion_score'] = shap_df.sum(axis=1)

    def categorize(score):
        if score > 2: return "highly suspicious"
        elif score > 0: return "medium suspicious"
        else: return "less suspicious"

    shap_df['suspicion_level'] = shap_df['suspicion_score'].apply(categorize)

    final_df = pd.concat([df_proc.reset_index(drop=True), shap_df[['suspicion_score','suspicion_level']]], axis=1)
    final_df['language'] = language
    return final_df

# -----------------------------
# 7️⃣ Train Models & Merge Outputs
# -----------------------------
model_hi, _ = train_model(df_hi, 'hi')
model_en, _ = train_model(df_en, 'en')

final_hi = build_final_output(df_hi, model_hi, language='hi')
final_en = build_final_output(df_en, model_en, language='en')

# Merge as continuation
final_combined = pd.concat([final_hi, final_en], ignore_index=True)

# -----------------------------
# 🔍 Show just few rows preview
# -----------------------------
print("✅ Combined output preview (Hindi + English):\n")
print(final_combined[['sender','subject','domain','keyword_suspicion','suspicion_score','suspicion_level','language']].head(15))


In [ ]:
# ================================
# Predict & Explain Single Email  **********
# ================================

def predict_email(email_dict, model, language='hi'):
    """
    email_dict should be a dictionary with keys:
    'sender', 'receiver', 'date', 'subject', 'body', 'urls'
    """
    df_new = pd.DataFrame([email_dict])
    df_new = preprocess_data(df_new, language)

    features = ['domain_suspicion','time_suspicion','keyword_suspicion',
                'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
                'subject_sentiment','body_sentiment','sender_reputation']
    X_new = df_new[features]

    # Prediction
    pred_prob = model.predict_proba(X_new)[0][1]  # probability of phishing
    pred_class = model.predict(X_new)[0]

    # SHAP explanation
    explainer = shap.Explainer(model)
    shap_values = explainer(X_new)

    shap_df = pd.DataFrame(shap_values.values, columns=X_new.columns)
    shap_df['feature_contribution'] = shap_df.sum(axis=1)

    # Categorize suspicion
    def categorize(score):
        if score>2: return "highly suspicious"
        elif score>0: return "medium suspicious"
        else: return "less suspicious"

    shap_df['suspicion_level'] = shap_df['feature_contribution'].apply(categorize)

    # Print results
    print("🔹 Prediction:", "Phishing" if pred_class==1 else "Safe")
    print("🔹 Phishing Probability:", round(pred_prob,4))
    print("🔹 Suspicion Level (SHAP):", shap_df['suspicion_level'].iloc[0])

    # Optional: plot SHAP force plot
    shap.force_plot(explainer(X_new)[0], matplotlib=True)

    return pred_class, pred_prob, shap_df

# ================================
# Example Usage: Hindi Phishing Email
# ================================
hindi_email = {
    'sender': 'उदाहरण@फ्रीइनाम.win',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 03:00:00',
    'subject': 'अभी अपना इनाम प्राप्त करें!',
    'body': 'यहाँ क्लिक करें और अपना खाता सत्यापित करें ताकि आप बड़ा इनाम जीत सकें!',
    'urls': 'http://फ्रीइनाम.win/claim'
}

predict_email(hindi_email, model_hi, 'hi')

# ================================
# Example Usage: English Phishing Email
# ================================
english_email = {
    'sender': 'winner@lotteryfree.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 02:00:00',
    'subject': 'You have won a prize!',
    'body': 'Click here to claim your free lottery reward now!',
    'urls': 'http://lotteryfree.com/claim'
}

predict_email(english_email, model_en, 'en')

# ================================
# Example Usage: Multilingual Model
# ================================
predict_email(hindi_email, model_multi, 'hi')
predict_email(english_email, model_multi, 'en')


In [ ]:
# -----------------------------
# Hindi Emails*******
# -----------------------------
hindi_phish = {
    'sender': 'उदाहरण@फ्रीइनाम.win',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 03:00:00',
    'subject': 'अभी अपना इनाम प्राप्त करें!',
    'body': 'यहाँ क्लिक करें और अपना खाता सत्यापित करें ताकि आप बड़ा इनाम जीत सकें!',
    'urls': 'http://फ्रीइनाम.win/claim'
}

hindi_safe = {
    'sender': 'support@statebankofindia.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 11:30:00',
    'subject': 'आपके खाते की मासिक सूचना',
    'body': 'प्रिय ग्राहक, आपके खाते का बैलेंस और लेन-देन का सारांश संलग्न है। किसी लिंक पर क्लिक करने की आवश्यकता नहीं है।',
    'urls': ''
}

# -----------------------------
# English Emails
# -----------------------------
english_phish = {
    'sender': 'winner@lotteryfree.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 02:00:00',
    'subject': 'You have won a prize!',
    'body': 'Click here to claim your free lottery reward now!',
    'urls': 'http://lotteryfree.com/claim'
}

english_safe = {
    'sender': 'support@chase.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 10:00:00',
    'subject': 'Your monthly account statement',
    'body': 'Dear Customer, your account summary is attached. No action is required.',
    'urls': ''
}

# -----------------------------
# Predictions using Separate Models
# -----------------------------
print("✅ Hindi Model Predictions")
predict_email(hindi_phish, model_hi, 'hi')
predict_email(hindi_safe, model_hi, 'hi')

print("\n✅ English Model Predictions")
predict_email(english_phish, model_en, 'en')
predict_email(english_safe, model_en, 'en')

# -----------------------------
# Predictions using Multilingual Model
# -----------------------------
print("\n✅ Multilingual Model Predictions")
predict_email(hindi_phish, model_multi, 'hi')
predict_email(hindi_safe, model_multi, 'hi')
predict_email(english_phish, model_multi, 'en')
predict_email(english_safe, model_multi, 'en')


In [ ]:
#SYSTEM ARCHITECTURE
# Multilingual Phishing Detection Pipeline (Hindi + English)*********
# =========================

import pandas as pd
import numpy as np
import re
from collections import Counter
from textblob import TextBlob
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.pyplot as plt

# -----------------------------
# 1️⃣ Load Datasets
# -----------------------------
df_hi = pd.read_excel("/content/hindi_emails_merged.xlsx")
df_en = pd.read_excel("/content/Cleaned_English_Dataset.xlsx")

# -----------------------------
# 2️⃣ Shared Helper Functions
# -----------------------------
trusted_domains = ['gmail.com','yahoo.com','hotmail.com']

def extract_domain(email):
    try:
        return email.split('@')[-1] if isinstance(email,str) else 'unknown'
    except:
        return 'unknown'

def classify_domain(domain):
    if domain in trusted_domains: return 0
    suspicious_tlds = ['.ru','.cn','.xyz','.top','.icu','.info','.tk','.ml','.ga','.cf']
    if any(tld in domain for tld in suspicious_tlds): return 2
    return 1

def classify_time(date_string):
    try:
        hour = pd.to_datetime(date_string).hour
        return 1 if 0<=hour<=5 or hour>=22 else 0
    except: return 0

def email_length(text): return len(text) if isinstance(text,str) else 0
def count_hyperlinks(text): return len(re.findall(r'http[s]?://', str(text))) if isinstance(text,str) else 0
def analyze_sentiment(text):
    try: return TextBlob(str(text)).sentiment.polarity
    except: return 0

def sender_reputation(sender_email):
    if any(domain in sender_email for domain in trusted_domains): return 0
    suspicious_domains = ['.ru','.cn','.xyz','.top','.icu','.info','.tk','.ml','.ga','.cf']
    if any(domain in sender_email for domain in suspicious_domains): return 1
    return 0.5

def has_keywords(text, keyword_list):
    try:
        text = str(text).lower()
        for word in keyword_list:
            if word.lower() in text: return 1
        return 0
    except: return 0

# NEW FEATURE: domain_age_suspicion (dummy heuristic)
def domain_age_suspicion(domain):
    if domain == "unknown":
        return 1
    if domain in trusted_domains:
        return 0
    return 0.5

# -----------------------------
# 3️⃣ Extract Dynamic Phishing Keywords
# -----------------------------
def extract_keywords(df, top_n=60):
    if 'label' in df.columns:
        phishing_texts = df[df['label']==1]['subject'].astype(str).tolist() + \
                         df[df['label']==1]['body'].astype(str).tolist()
    else:
        phishing_texts = df['subject'].astype(str).tolist() + df['body'].astype(str).tolist()
    words = []
    for text in phishing_texts: words.extend(re.findall(r'\w+', text))
    return [w for w,_ in Counter(words).most_common(top_n)]

PHISHING_KEYWORDS_HINDI = list(set(extract_keywords(df_hi,60) + [
    "पासवर्ड","तुरंत","यहाँ क्लिक करें","सत्यापित करें","लॉगिन","खाता",
    "बैंक","फ्री","ऑफ़र","जीतें","इनाम","अपडेट","भुगतान","रिचार्ज","सुरक्षा","संदेहास्पद","डाउनलोड"
]))
PHISHING_KEYWORDS_EN = list(set(extract_keywords(df_en,60) + [
    "lottery","prize","win","bonus","free","offer","password","verify",
    "account","click","login","update","cash","otp","urgent"
]))

# -----------------------------
# 4️⃣ Preprocessing Function
# -----------------------------
def preprocess_data(df, language='hi'):
    df = df.copy()
    df['sender'] = df['sender'].fillna('unknown').astype(str)
    df['subject'] = df['subject'].fillna('').astype(str)
    df['body'] = df['body'].fillna('').astype(str)
    df['urls'] = df['urls'].fillna('').astype(str)

    df['domain'] = df['sender'].apply(extract_domain)
    df['domain_suspicion'] = df['domain'].apply(classify_domain)
    df['domain_age_suspicion'] = df['domain'].apply(domain_age_suspicion)   # ✅ Added here
    df['time_suspicion'] = df['date'].apply(classify_time)

    df['keyword_suspicion_subject'] = df['subject'].apply(
        lambda x: has_keywords(x, PHISHING_KEYWORDS_HINDI if language=='hi' else PHISHING_KEYWORDS_EN)
    )
    df['keyword_suspicion_body'] = df['body'].apply(
        lambda x: has_keywords(x, PHISHING_KEYWORDS_HINDI if language=='hi' else PHISHING_KEYWORDS_EN)
    )
    df['keyword_suspicion'] = df[['keyword_suspicion_subject','keyword_suspicion_body']].max(axis=1)

    df['url_suspicion'] = df['urls'].apply(lambda x: 1 if x else 0)
    df['subject_length'] = df['subject'].apply(email_length)
    df['body_length'] = df['body'].apply(email_length)
    df['subject_hyperlink_count'] = df['subject'].apply(count_hyperlinks)
    df['body_hyperlink_count'] = df['body'].apply(count_hyperlinks)
    df['subject_sentiment'] = df['subject'].apply(analyze_sentiment)
    df['body_sentiment'] = df['body'].apply(analyze_sentiment)
    df['sender_reputation'] = df['sender'].apply(sender_reputation)

    # ✅ Only map labels if column exists
    if 'label' in df.columns:
        if df['label'].dtype==object:
            df['label'] = df['label'].map({'ham':0,'spam':1})

    return df

# -----------------------------
# 5️⃣ Train Model Function
# -----------------------------
def train_model(df, language='hi'):
    df = preprocess_data(df, language)
    df.fillna(0, inplace=True)

    features = ['domain_suspicion','domain_age_suspicion','time_suspicion','keyword_suspicion',
                'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
                'subject_sentiment','body_sentiment','sender_reputation']
    X = df[features]
    y = df['label']

    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)

    scale_pos_weight = y_res.value_counts().iloc[0] / y_res.value_counts().iloc[1]

    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                          scale_pos_weight=scale_pos_weight)
    model.fit(X_res, y_res)
    return model, X

# -----------------------------
# 6️⃣ SHAP Calculation Function
# -----------------------------
def calculate_shap(model, X):
    explainer = shap.Explainer(model)
    shap_values = explainer(X)
    shap_df = pd.DataFrame(shap_values.values, columns=X.columns)
    shap_df['suspicion_score'] = shap_df.sum(axis=1)

    def categorize(score):
        if score>2: return "highly suspicious"
        elif score>0: return "medium suspicious"
        else: return "less suspicious"

    shap_df['suspicion_level'] = shap_df['suspicion_score'].apply(categorize)
    return shap_df

# -----------------------------
# 7️⃣ Train Separate Models
# -----------------------------
model_hi, X_hi = train_model(df_hi, 'hi')
model_en, X_en = train_model(df_en, 'en')
shap_hi = calculate_shap(model_hi, X_hi)
shap_en = calculate_shap(model_en, X_en)
print("✅ Separate Hindi & English models trained successfully.")

# -----------------------------
# 8️⃣ Train Merged Multilingual Model
# -----------------------------
df_hi = preprocess_data(df_hi, 'hi')
df_en = preprocess_data(df_en, 'en')
df_merged = pd.concat([df_hi, df_en], ignore_index=True)
df_merged.fillna(0, inplace=True)

features = ['domain_suspicion','domain_age_suspicion','time_suspicion','keyword_suspicion',
            'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
            'subject_sentiment','body_sentiment','sender_reputation']
X = df_merged[features]
y = df_merged['label']

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

scale_pos_weight = y_res.value_counts().iloc[0] / y_res.value_counts().iloc[1]

model_multi = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                            scale_pos_weight=scale_pos_weight)
model_multi.fit(X_res, y_res)
shap_multi = calculate_shap(model_multi, X)

print("✅ Merged multilingual model trained successfully.")

# -----------------------------
# 9️⃣ Predict New Emails (Phishing & Non-Phishing)
# -----------------------------
def predict_email_full(email_dict, model, language='hi'):
    nd = pd.DataFrame([email_dict])
    nd = preprocess_data(nd, language)

    features = ['domain_suspicion','domain_age_suspicion','time_suspicion','keyword_suspicion',
                'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
                'subject_sentiment','body_sentiment','sender_reputation']
    nd_X = nd[features]

    # Prediction
    pred_prob = model.predict_proba(nd_X)[0][1]
    pred_class = model.predict(nd_X)[0]

    explainer = shap.Explainer(model)
    shap_values = explainer(nd_X)
    shap_df = pd.DataFrame(shap_values.values, columns=nd_X.columns)
    shap_df['feature_contribution'] = shap_df.sum(axis=1)

    def categorize(score):
        if score>2: return "highly suspicious"
        elif score>0: return "medium suspicious"
        else: return "less suspicious"

    shap_df['suspicion_level'] = shap_df['feature_contribution'].apply(categorize)

    print("🔹 Prediction:", "Phishing" if pred_class==1 else "Safe")
    print("🔹 Phishing Probability:", round(pred_prob,4))
    print("🔹 Suspicion Level (SHAP):", shap_df['suspicion_level'].iloc[0])

    return pred_class, pred_prob, shap_df


In [ ]:
# -----------------------------
# Predict and Show Full Output (Multilingual)**********(RESULTS AND DISCUSSION_PPT)
# -----------------------------
def predict_email_full(email_dict, model, language='hi'):
    """
    Predict phishing probability for a single email (Hindi or English)
    and return a DataFrame with features, prediction, probability, and SHAP suspicion level.
    """
    import pandas as pd

    # Convert dict -> DataFrame
    nd = pd.DataFrame([email_dict])

    # Preprocess
    nd = preprocess_data(nd, language)
    nd.fillna(0, inplace=True)

    # Define required features (MUST match training features of the model)
    features = [
        'domain_suspicion', 'domain_age_suspicion', 'time_suspicion', 'keyword_suspicion',
        'subject_length','body_length','subject_hyperlink_count','body_hyperlink_count',
        'subject_sentiment','body_sentiment','sender_reputation'
    ]

    # Ensure all features exist (fill missing with 0)
    for col in features:
        if col not in nd.columns:
            nd[col] = 0

    # Select features for model
    nd_X = nd[features]

    # Prediction
    prediction = model.predict(nd_X)[0]
    probability = model.predict_proba(nd_X)[0][1]
    print("Prediction (0=Safe,1=Phishing):", prediction)
    print("Suspicion Probability:", round(probability,4))

    # SHAP values
    shap_df = calculate_shap(model, nd_X)

    # Merge SHAP suspicion level with original data
    nd = pd.concat([nd, shap_df[['suspicion_level']]], axis=1)

    return nd


# -----------------------------
# Hindi Emails (Phishing & Non-Phishing)
# -----------------------------
hindi_phish = {
    'sender': 'उदाहरण@फ्रीइनाम.win',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 03:00:00',
    'subject': 'अभी अपना इनाम प्राप्त करें!',
    'body': 'यहाँ क्लिक करें और अपना खाता सत्यापित करें ताकि आप बड़ा इनाम जीत सकें!',
    'urls': 'http://फ्रीइनाम.win/claim'
}

hindi_safe = {
    'sender': 'support@statebankofindia.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 11:30:00',
    'subject': 'आपके खाते की मासिक सूचना',
    'body': 'प्रिय ग्राहक, आपके खाते का बैलेंस और लेन-देन का सारांश संलग्न है। किसी लिंक पर क्लिक करने की आवश्यकता नहीं है।',
    'urls': ''
}


# -----------------------------
# English Emails (Phishing & Non-Phishing)
# -----------------------------
english_phish = {
    'sender': 'winner@lotteryfree.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 02:00:00',
    'subject': 'You have won a prize!',
    'body': 'Click here to claim your free lottery reward now!',
    'urls': 'http://lotteryfree.com/claim'
}

english_safe = {
    'sender': 'support@chase.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 10:00:00',
    'subject': 'Your monthly account statement',
    'body': 'Dear Customer, your account summary is attached. No action is required.',
    'urls': ''
}


# -----------------------------
# Run Predictions for Hindi Model
# -----------------------------
print("✅ Hindi Model Predictions")
nd_hi_phish = predict_email_full(hindi_phish, model_hi, 'hi')
nd_hi_safe = predict_email_full(hindi_safe, model_hi, 'hi')

# -----------------------------
# Run Predictions for English Model
# -----------------------------
print("\n✅ English Model Predictions")
nd_en_phish = predict_email_full(english_phish, model_en, 'en')
nd_en_safe = predict_email_full(english_safe, model_en, 'en')

# -----------------------------
# Run Predictions for Multilingual Model
# -----------------------------
#print("\n✅ Multilingual Model Predictions")
nd_multi_hi_phish = predict_email_full(hindi_phish, model_multi, 'hi')
nd_multi_hi_safe = predict_email_full(hindi_safe, model_multi, 'hi')
nd_multi_en_phish = predict_email_full(english_phish, model_multi, 'en')
nd_multi_en_safe = predict_email_full(english_safe, model_multi, 'en')


# -----------------------------
# Optional: Combine All Results in One DataFrame
# -----------------------------
all_results = pd.concat([
    nd_hi_phish.assign(model='Hindi'),
    nd_hi_safe.assign(model='Hindi'),
    nd_en_phish.assign(model='English'),
    nd_en_safe.assign(model='English'),
    #nd_multi_hi_phish.assign(model='Multilingual'),
    #nd_multi_hi_safe.assign(model='Multilingual'),
    #nd_multi_en_phish.assign(model='Multilingual'),
    #nd_multi_en_safe.assign(model='Multilingual')
], ignore_index=True)

print("\n✅ Combined Results (Phishing + Non-Phishing, Hindi & English):")
print(all_results.head(10))   # only show first 10 rows


In [ ]:
# -----------------------------
# Predict and Show Full Output (Hindi + English Only)
# -----------------------------
def predict_email_full(email_dict, model, language='hi'):
    """
    Predict phishing probability for a single email (Hindi or English)
    and return a DataFrame with features, prediction, probability, and SHAP suspicion level.
    """
    import pandas as pd

    # Convert dict -> DataFrame
    nd = pd.DataFrame([email_dict])

    # Preprocess
    nd = preprocess_data(nd, language)
    nd.fillna(0, inplace=True)

    # Define required features (MUST match training features of the model)
    features = [
        'domain_suspicion', 'domain_age_suspicion', 'time_suspicion', 'keyword_suspicion',
        'subject_length', 'body_length', 'subject_hyperlink_count', 'body_hyperlink_count',
        'subject_sentiment', 'body_sentiment', 'sender_reputation'
    ]

    # Ensure all features exist (fill missing with 0)
    for col in features:
        if col not in nd.columns:
            nd[col] = 0

    # Select features for model
    nd_X = nd[features]

    # Prediction
    prediction = model.predict(nd_X)[0]
    probability = model.predict_proba(nd_X)[0][1]
    print("Prediction (0=Safe,1=Phishing):", prediction)
    print("Suspicion Probability:", round(probability, 4))

    # SHAP values
    shap_df = calculate_shap(model, nd_X)

    # Merge SHAP suspicion level with original data
    nd = pd.concat([nd, shap_df[['suspicion_level']]], axis=1)

    return nd


# -----------------------------
# Hindi Emails (Phishing & Non-Phishing)
# -----------------------------
hindi_phish = {
    'sender': 'उदाहरण@फ्रीइनाम.win',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 03:00:00',
    'subject': 'अभी अपना इनाम प्राप्त करें!',
    'body': 'यहाँ क्लिक करें और अपना खाता सत्यापित करें ताकि आप बड़ा इनाम जीत सकें!',
    'urls': 'http://फ्रीइनाम.win/claim'
}

hindi_safe = {
    'sender': 'support@statebankofindia.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 11:30:00',
    'subject': 'आपके खाते की मासिक सूचना',
    'body': 'प्रिय ग्राहक, आपके खाते का बैलेंस और लेन-देन का सारांश संलग्न है। किसी लिंक पर क्लिक करने की आवश्यकता नहीं है।',
    'urls': ''
}


# -----------------------------
# English Emails (Phishing & Non-Phishing)
# -----------------------------
english_phish = {
    'sender': 'winner@lotteryfree.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 02:00:00',
    'subject': 'You have won a prize!',
    'body': 'Click here to claim your free lottery reward now!',
    'urls': 'http://lotteryfree.com/claim'
}

english_safe = {
    'sender': 'support@chase.com',
    'receiver': 'user@gmail.com',
    'date': '2025-06-18 10:00:00',
    'subject': 'Your monthly account statement',
    'body': 'Dear Customer, your account summary is attached. No action is required.',
    'urls': ''
}


# -----------------------------
# Run Predictions for Hindi Model
# -----------------------------
print("✅ Hindi Model Predictions")
nd_hi_phish = predict_email_full(hindi_phish, model_hi, 'hi')
nd_hi_safe = predict_email_full(hindi_safe, model_hi, 'hi')


# -----------------------------
# Run Predictions for English Model
# -----------------------------
print("\n✅ English Model Predictions")
nd_en_phish = predict_email_full(english_phish, model_en, 'en')
nd_en_safe = predict_email_full(english_safe, model_en, 'en')


# -----------------------------
# Combine Results (Hindi + English only)
# -----------------------------
import pandas as pd
all_results = pd.concat([
    nd_hi_phish.assign(model='Hindi'),
    nd_hi_safe.assign(model='Hindi'),
    nd_en_phish.assign(model='English'),
    nd_en_safe.assign(model='English')
], ignore_index=True)

print("\n✅ Combined Results (Phishing + Non-Phishing, Hindi & English):")
print(all_results.head(10))  # show first 10 rows
